## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods or enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# Loading Groq API 
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [2]:
# Importing the ChatGroq class from langchain_groq module
from langchain_groq import ChatGroq

## Initializing the GROQ with Qwen 3 32B model
model=ChatGroq(model="qwen/qwen3-32b")

In [16]:
## Checking model details
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: str = Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7c62ae2d9d30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c62ae2daa50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'string'}}, 'required': ['title', 'year', 'director'

In [5]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating='8.8')

In [14]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact Information for the person."""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email of the contact")
    phone: str = Field(..., description="The phone number of the contact")


agent=create_agent(model=model, response_format=ContactInfo)

result=agent.invoke({
    "messages":[{"role": "user", "content": "Extract the contact information from the following text: 'John Doe can be reached at john.doe@example.com or 123-456-7890"}]
    })

print(result)

{'messages': [HumanMessage(content="Extract the contact information from the following text: 'John Doe can be reached at john.doe@example.com or 123-456-7890", additional_kwargs={}, response_metadata={}, id='3dfa2cb1-399e-4d65-b4a4-45be8741b892'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The text says, "John Doe can be reached at john.doe@example.com or 123-456-7890." I need to identify the name, email, and phone number here.\n\nFirst, the name is clearly "John Doe." Then, the email address is "john.doe@example.com," which follows the typical email format. The phone number is "123-456-7890," which is in the standard US format with hyphens. \n\nI should check if all required fields are present. The function ContactInfo requires name, email, and phone. All three are here. No missing data. I need to make sure there\'s no extra information or formatting issues. The phone number mig

In [15]:
result['structured_response']

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

In [6]:
## Message output along with the parsed output
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: str = Field(..., description="The movies rating out of 10")


model_with_structure=model.with_structured_output(Movie, include_raw=True)
model_with_structure.invoke("Provide details about the movie Inception")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, year, director, and rating. I need to fill those in. I remember Inception was directed by Christopher Nolan. It came out in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and director. Yep, 2010 and Christopher Nolan. The rating is 8.8. So I'll structure the tool call with those parameters.\n", 'tool_calls': [{'id': 'm458q6z8d', 'function': {'arguments': '{"director":"Christopher Nolan","rating":"8.8","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 163, 'prompt_tokens': 230, 'total_tokens': 393, 'completion_time': 0.256327905, 'completion_tokens_details': {'reasoning_tokens': 115}, 'prompt_time': 0.009017859, 'prompt_tokens_details': None, 'queue_time': 0.158957

In [7]:
## Nested structured output
class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")

class MovieDetails(BaseModel):
    title: str 
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_structure=model.with_structured_output(MovieDetails)
respose=model_with_structure.invoke("Provide details about the movie Inception")
respose

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Tom Hardy', role='Bane')], genre=['Sci-Fi', 'Action', 'Thriller'], budget=None)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [8]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[str, ..., "The movies rating out of 10"]


model_with_TypedDict=model.with_structured_output(MovieDict)
model_with_TypedDict.invoke("Provide details about the movie Avengers: Endgame")

{'director': 'Anthony Russo, Joe Russo',
 'rating': '8.4',
 'title': 'Avengers: Endgame',
 'year': 2019}

In [9]:
## Nested structured output
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str 
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in Millions USD")

model_with_TypedDict=model.with_structured_output(MovieDetails)
model_with_TypedDict.invoke("Provide details about the movie Avengers: Endgame")

{'budget': 356000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Jon Favreau', 'role': 'Happy Hogan'}],
 'genre': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers: Endgame',
 'year': 2019}

### DataClasses

A data class is a class ypically containing mainly data, although there aren't really any restriction. We can create it using the @dataclass decorator

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ContactInfo:
    """Contact Information for the person."""
    name: str # The name of the contact
    email: str #The email of the contact
    phone: str # The phone number of the contact

{'messages': [HumanMessage(content="Extract the contact information from the following text: 'John Doe can be reached at john.doe@example.com or 123-456-7890", additional_kwargs={}, response_metadata={}, id='4838ac6f-587d-4816-a727-805fecff65c9'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The text is: "John Doe can be reached at john.doe@example.com or 123-456-7890". \n\nFirst, I need to identify the different components. The name is clearly "John Doe". Then there\'s an email address: john.doe@example.com. The phone number is 123-456-7890.\n\nNow, looking at the tool provided, the ContactInfo function requires name, email, and phone. All three are present here. I should make sure the email is correctly formatted and the phone number is in the right format. The phone number here is in the format with hyphens, which is standard. \n\nI don\'t see any additional contact methods, lik

In [17]:
agent=create_agent(model=model, response_format=ContactInfo)

result=agent.invoke({
    "messages":[{"role": "user", "content": "Extract the contact information from the following text: 'John Doe can be reached at john.doe@example.com or 123-456-7890"}]
    })

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content="Extract the contact information from the following text: 'John Doe can be reached at john.doe@example.com or 123-456-7890", additional_kwargs={}, response_metadata={}, id='8ba4820d-26fc-4ba6-adca-3df6818cc8dd'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The text is "John Doe can be reached at john.doe@example.com or 123-456-7890". I need to identify the name, email, and phone number.\n\nFirst, the name is clearly "John Doe". Then the email is "john.doe@example.com" and the phone number is "123-456-7890". The function ContactInfo requires all three fields: name, email, and phone. I should make sure all are included in the arguments. Let me check the parameters again. Yes, the required fields are there. So I need to structure a JSON object with these three pieces of information. No need to call any other functions since the information is straig